# 1차 ↔ 2차 데이터 연결 테이블

대회 제공 데이터(2차 가공, `moldset_*_cn7/rg3.csv`)를 분석의 기본으로 두고, 2차에서 해결할 수 없는 부분만 1차 가공 데이터에서 가져와 붙입니다.

| 가져오는 정보 | 1차 컬럼 | 쓰는 곳 | 모델 입력 |
|---|---|---|---|
| 원래 단위 공정값 | 공정변수 24개 (+ `Switch_Over_Position`) | 물리적 파생변수 | ✅ |
| 캐비티(LH/RH) | `PART_NAME` | 쌍 구조 확정, 캐비티별 분석 | ❌ 분석용 |
| 시간 정보 | `TimeStamp`, `PART_FACT_PLAN_DATE` | 시간 순서 분할, 당일 샷 순번, 직전 샷 대비 변화 | ✅ (순번·변화량) |
| 불량 유형 | `Reason` | 실패 조건 분석 | ❌ **입력 금지** (검사 후 정보) |
| 일별 불량 수 | `ERR_FACT_QTY` (unlabeled, 있으면) | pseudo-label 검증 | ❌ 검증용 |

**연결 방법**: 1차에서 해당 행을 골라 가이드북과 같은 방식(파일별 StandardScaler, ddof=0)으로 표준화한 뒤, 2차 값과 **완전히 일치하는지** 확인하고 붙입니다.

**산출물** (`OUTPUT_DIR`)
- `linked_labeled.csv`: labeled 1차 CN7·RG3 전체 행 + 2차 인덱스(`sec_index`) + 가져온 컬럼
- `linked_unlabeled_cn7.csv`, `linked_unlabeled_rg3.csv`: 2차 unlabeled 행 순서 그대로 + 가져온 컬럼

## 1. 설정

In [ ]:
from pathlib import Path
root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'notebooks/157_rg3_existing_connected_pipeline_runner.ipynb').exists())
DATA_DIR = str(root/'data/raw_input')
OUTPUT_DIR = str(root/'data/taein/linked')
RUN_UNLABELED = True   # unlabeled_data.csv(약 225MB) 연결 여부

import os
import numpy as np
import pandas as pd
os.makedirs(OUTPUT_DIR, exist_ok=True)
P = lambda f: os.path.join(DATA_DIR, f)

# 2차 공정변수 24개 (2차 파일 컬럼 순서 그대로)
FEATURES = [c for c in pd.read_csv(P('moldset_labeled_cn7.csv'), nrows=1).columns
            if c not in ('Unnamed: 0', 'PassOrFail')]
print(len(FEATURES), FEATURES)

In [ ]:
def zscore(df):
    # 가이드북 [코드 43]: StandardScaler (모집단 표준편차, ddof=0). 상수 컬럼은 0으로
    x = df[FEATURES].astype(float)
    sd = x.std(ddof=0).replace(0, 1)
    return (x - x.mean()) / sd

def max_abs_diff(a, b):
    return float(np.nanmax(np.abs(a.values - b.values)))

PRODUCT_PREFIX = {'cn7': 'CN7', 'rg3': 'RG3'}

## 2. Labeled 연결

1차 `moldset_labeled.csv`(2,607행)에서 제품별로 행을 골라 2차와 비교합니다.

확인된 규칙
- **CN7**: 생산일(`PART_FACT_PLAN_DATE`) 2020-10-27을 **제외**한 1,211행이 2차와 일치 (10-27 214행은 2차에 없음)
- **RG3**: 1,182행 전부가 2차와 일치
- 2차 `Unnamed: 0` = CN7 → RG3 순서로 이어 붙인 뒤의 행 번호

In [ ]:
M = pd.read_csv(P('moldset_labeled.csv'), low_memory=False)
M['PassOrFail'] = M['PassOrFail'].map({'Y': 0, 'N': 1}).fillna(M['PassOrFail']).astype(int) \
    if M['PassOrFail'].dtype == object else M['PassOrFail']
M['plan_date'] = pd.to_datetime(M['PART_FACT_PLAN_DATE'].str[:10])
M['TimeStamp'] = pd.to_datetime(M['TimeStamp'])
M['product'] = M['PART_NAME'].str[:3].str.lower()
M['cavity'] = M['PART_NAME'].str[-2:]           # LH / RH
M['row_1st'] = np.arange(len(M))               # 1차 파일 행 번호

EXCLUDE_DATES = {'cn7': ['2020-10-27'], 'rg3': []}

parts, offset, report = [], 0, []
for p in ['cn7', 'rg3']:
    sec = pd.read_csv(P(f'moldset_labeled_{p}.csv'))
    prim = M[M['product'] == p].copy()
    prim['in_secondary'] = ~prim['plan_date'].isin(pd.to_datetime(EXCLUDE_DATES[p]))
    kept = prim[prim['in_secondary']]

    diff = max_abs_diff(zscore(kept).reset_index(drop=True), sec[FEATURES])
    label_ok = (kept['PassOrFail'].values == sec['PassOrFail'].values).mean()
    report.append({'product': p, '1차 행': len(prim), '2차 행': len(sec), '연결 행': len(kept),
                   '값 최대 차이': diff, '라벨 일치율': label_ok})
    assert len(kept) == len(sec) and diff < 1e-6 and label_ok == 1.0, f'{p} 연결 실패'

    prim['sec_index'] = np.nan
    prim.loc[kept.index, 'sec_index'] = sec['Unnamed: 0'].values
    parts.append(prim)

L = pd.concat(parts)
pd.DataFrame(report)

### 2-1. 샷(LH/RH 쌍) 확정

같은 `TimeStamp`에 LH·RH가 하나씩 있고 공정값이 같은지 확인합니다.

In [ ]:
g = L.groupby(['product', 'TimeStamp']).agg(n=('cavity', 'size'),
                                            cav=('cavity', lambda s: ''.join(sorted(s))),
                                            n_values=(FEATURES[0], 'nunique'))
print(g.groupby('product')['cav'].value_counts().to_string())
print('같은 TimeStamp 안에서 공정값이 다른 샷:', int((g['n_values'] > 1).sum()))

L['shot_id'] = L.groupby(['product', 'TimeStamp']).ngroup()
L['shot_fail'] = L.groupby('shot_id')['PassOrFail'].transform('max')
L['shot_fail_n'] = L.groupby('shot_id')['PassOrFail'].transform('sum')

shot = L.drop_duplicates('shot_id')
print()
print(shot.groupby('product').agg(샷=('shot_id', 'size'), 불량샷=('shot_fail', 'sum'),
                                  양쪽불량=('shot_fail_n', lambda s: (s == 2).sum())).to_string())
print()
print('캐비티별 불량 (2차 범위):')
print(L[L['in_secondary']].groupby(['product', 'cavity'])['PassOrFail'].sum().unstack().to_string())

### 2-2. 시간 파생 정보

- `shot_seq_in_day`: 같은 날(TimeStamp 기준) 몇 번째 샷인지
- `prev_gap_sec`: 직전 샷과의 시간 간격(초). 작업 중단 후 재시작 구간을 찾는 데 사용

In [ ]:
L = L.sort_values(['product', 'TimeStamp', 'cavity']).copy()
L['ts_date'] = L['TimeStamp'].dt.normalize()
shot = L.drop_duplicates('shot_id')[['shot_id', 'product', 'TimeStamp', 'ts_date']].copy()
shot['shot_seq_in_day'] = shot.groupby(['product', 'ts_date']).cumcount() + 1
shot['prev_gap_sec'] = shot.groupby('product')['TimeStamp'].diff().dt.total_seconds()
L = L.merge(shot[['shot_id', 'shot_seq_in_day', 'prev_gap_sec']], on='shot_id', how='left')

print(L.drop_duplicates('shot_id').groupby(['product', 'ts_date']).size().rename('샷 수').to_string())

### 2-3. 저장

원래 단위 공정값은 `_raw` 접미사를 붙여 저장합니다. 2차 표준화 값은 2차 파일을 그대로 쓰면 됩니다.

In [ ]:
RAW_COLS = FEATURES + ['Switch_Over_Position']
keep = (['product', 'sec_index', 'in_secondary', 'row_1st', '_id', 'TimeStamp', 'plan_date',
         'cavity', 'shot_id', 'shot_seq_in_day', 'prev_gap_sec', 'PassOrFail', 'shot_fail', 'Reason'] + RAW_COLS)
out = L[keep].rename(columns={c: f'{c}_raw' for c in RAW_COLS}).sort_values(['product', 'row_1st'])
out.to_csv(os.path.join(OUTPUT_DIR, 'linked_labeled.csv'), index=False, encoding='utf-8-sig')
print(out.shape)
print('2차 범위 밖 행 (in_secondary=False):', int((~out['in_secondary']).sum()),
      '| 그중 불량:', int(out.loc[~out['in_secondary'], 'PassOrFail'].sum()))
out.head()

## 3. Unlabeled 연결

가설: 2차 unlabeled의 `Unnamed: 0` = 1차 `unlabeled_data.csv`의 행 번호

- **방법 A**: 그 행 번호로 1차 행을 바로 가져와 표준화 후 비교
- **방법 B** (A 실패 시): 1차에서 제품명으로 행을 골라 순서대로 비교

`unlabeled_data.csv`는 약 225MB라 필요한 컬럼만 읽습니다.

In [ ]:
if RUN_UNLABELED:
    head = pd.read_csv(P('unlabeled_data.csv'), nrows=5)
    print(list(head.columns))
    META = [c for c in ['_id', 'TimeStamp', 'PART_FACT_PLAN_DATE', 'PART_NAME', 'EQUIP_NAME', 'ERR_FACT_QTY']
            if c in head.columns]
    USE = META + [c for c in RAW_COLS if c in head.columns]
    U = pd.read_csv(P('unlabeled_data.csv'), usecols=USE, low_memory=False)
    print(U.shape)
    print(U['PART_NAME'].value_counts().head(10).to_string() if 'PART_NAME' in U else '')

In [ ]:
def link_unlabeled(p):
    sec = pd.read_csv(P(f'moldset_unlabeled_{p}.csv'))
    idx = sec['Unnamed: 0'].values

    # 방법 A: 행 번호로 직접 연결
    if idx.max() < len(U):
        cand = U.iloc[idx]
        diff = max_abs_diff(zscore(cand).reset_index(drop=True), sec[FEATURES])
        name_ok = cand['PART_NAME'].str.upper().str.startswith(PRODUCT_PREFIX[p]).mean() if 'PART_NAME' in cand else np.nan
        print(f'[{p}] 방법 A | 값 최대 차이 {diff:.2e} | 제품명 일치율 {name_ok}')
        if diff < 1e-6:
            return cand.assign(sec_index=idx, link_method='A'), sec

    # 방법 B: 제품명으로 골라 순서대로 비교
    cand = U[U['PART_NAME'].str.upper().str.startswith(PRODUCT_PREFIX[p])]
    print(f'[{p}] 방법 B | 1차 후보 {len(cand)}행, 2차 {len(sec)}행')
    if len(cand) == len(sec):
        diff = max_abs_diff(zscore(cand).reset_index(drop=True), sec[FEATURES])
        print(f'[{p}] 방법 B | 값 최대 차이 {diff:.2e}')
        if diff < 1e-6:
            return cand.assign(sec_index=idx, link_method='B'), sec
    raise RuntimeError(f'{p}: 자동 연결 실패 — 위 출력(행 수, 차이)을 확인하세요')

if RUN_UNLABELED:
    for p in ['cn7', 'rg3']:
        cand, sec = link_unlabeled(p)
        uo = cand.copy()
        uo['TimeStamp'] = pd.to_datetime(uo['TimeStamp'])
        if 'PART_NAME' in uo:
            uo['cavity'] = uo['PART_NAME'].str[-2:]
        uo = uo.rename(columns={c: f'{c}_raw' for c in RAW_COLS if c in uo})
        first = ['sec_index', 'link_method'] + [c for c in ['_id', 'TimeStamp', 'PART_FACT_PLAN_DATE', 'cavity', 'ERR_FACT_QTY'] if c in uo]
        uo = uo[first + [c for c in uo.columns if c not in first + ['PART_NAME', 'EQUIP_NAME']]]
        uo.to_csv(os.path.join(OUTPUT_DIR, f'linked_unlabeled_{p}.csv'), index=False, encoding='utf-8-sig')
        print(p, uo.shape, '저장 완료')

## 4. 연결 결과 요약

| 항목 | 내용 |
|---|---|
| labeled CN7 | 1차 1,425행 중 **생산일 2020-10-27을 뺀 1,211행**이 2차와 완전히 일치 |
| labeled RG3 | 1차 1,182행 전부 2차와 완전히 일치 |
| LH/RH 쌍 | 같은 `TimeStamp`에 LH·RH가 하나씩, 공정값 동일 → **샷 단위 구조 확정** |
| unlabeled | 3장 출력 참고 (방법 A/B 중 어느 것으로 연결됐는지) |

**활용 시 주의**
- `Reason`, `shot_fail`, `ERR_FACT_QTY`는 검사 후 정보 → **모델 입력 금지**, 분석·검증에만 사용
- `in_secondary=False` 행(CN7 10-27)은 대회 제공 범위 밖. 쓰려면 별도로 사용 이유를 설명해야 함

## 5. 추가 확인

EDA 전에 데이터 범위를 확정하기 위한 세 가지 확인입니다. 3장(unlabeled 연결)을 실행한 뒤에 돌려야 합니다 (`U` 변수 사용).

1. **2차 unlabeled 선택 기준**: 1차의 같은 제품 행 중 어떤 기준으로 2차에 들어갔는가
2. **labeled ↔ unlabeled 중복**: 같은 샷이 양쪽에 다 있는가 (준지도 학습 시 평가 누수 위험)
3. **`ERR_FACT_QTY` 상태**: 값이 제대로 붙었는가, 어떤 단위(샷/일/계획)로 기록되는가

### 5-1. 2차 unlabeled 선택 기준

1차에서 같은 제품명을 가진 행을 **2차에 들어간 행 / 빠진 행**으로 나눠서 설비·생산일·날짜별로 비교합니다.
한쪽에만 나타나는 값이 있으면 그게 선택 기준입니다.

In [ ]:
U['TimeStamp_dt'] = pd.to_datetime(U['TimeStamp'], errors='coerce')
U['ts_date'] = U['TimeStamp_dt'].dt.date

sel_report = []
for p in ['cn7', 'rg3']:
    sec = pd.read_csv(P(f'moldset_unlabeled_{p}.csv'), usecols=['Unnamed: 0'])
    same = U[U['PART_NAME'].str.upper().str.startswith(PRODUCT_PREFIX[p])].copy()
    same['in_2nd'] = same.index.isin(sec['Unnamed: 0'])
    print(f'==================== {p} ====================')
    print(f"1차 같은 제품 {len(same):,}행 → 2차 포함 {same['in_2nd'].sum():,}행 / 제외 {(~same['in_2nd']).sum():,}행")

    for col in [c for c in ['EQUIP_NAME', 'PART_NAME', 'PART_FACT_PLAN_DATE', 'ts_date'] if c in same]:
        tab = pd.crosstab(same[col], same['in_2nd'].map({True: '2차 포함', False: '제외'}))
        print(f'\n[{col}]')
        print(tab.to_string())

    # 포함 행이 1차 파일에서 연속 구간인지
    idx = np.sort(sec['Unnamed: 0'].values)
    blocks = int((np.diff(idx) > 1).sum()) + 1
    print(f'\n2차 포함 행의 1차 행 번호: {idx.min():,} ~ {idx.max():,}, 끊긴 구간 수 {blocks}')
    print('2차 포함 기간:', same.loc[same['in_2nd'], 'TimeStamp_dt'].min(), '~', same.loc[same['in_2nd'], 'TimeStamp_dt'].max())
    print('제외 기간   :', same.loc[~same['in_2nd'], 'TimeStamp_dt'].min(), '~', same.loc[~same['in_2nd'], 'TimeStamp_dt'].max())
    print()

### 5-2. labeled ↔ unlabeled 중복

두 가지 기준으로 봅니다.
- **시간 기준**: 같은 제품·같은 `TimeStamp`·같은 캐비티
- **값 기준**: 원래 단위 공정값 24개가 모두 같음

중복이 있으면 준지도 학습에서 **평가용 labeled 샷이 unlabeled로 학습에 들어갈 수 있으니** 제거해야 합니다.

In [ ]:
LL = pd.read_csv(os.path.join(OUTPUT_DIR, 'linked_labeled.csv'), parse_dates=['TimeStamp'])
RAW = [f'{c}_raw' for c in FEATURES]

for p in ['cn7', 'rg3']:
    UL = pd.read_csv(os.path.join(OUTPUT_DIR, f'linked_unlabeled_{p}.csv'), parse_dates=['TimeStamp'])
    lab = LL[LL['product'] == p]
    print(f'==================== {p} ====================')
    print('labeled   기간:', lab['TimeStamp'].min(), '~', lab['TimeStamp'].max())
    print('unlabeled 기간:', UL['TimeStamp'].min(), '~', UL['TimeStamp'].max())

    keys = ['TimeStamp', 'cavity'] if 'cavity' in UL else ['TimeStamp']
    t_dup = lab.merge(UL[keys].drop_duplicates(), on=keys, how='inner')
    v_dup = lab[RAW].round(4).merge(UL[RAW].round(4).drop_duplicates(), on=RAW, how='inner')
    print(f'시간 기준 중복: labeled {len(t_dup)}행 (2차 범위 {int(t_dup["in_secondary"].sum())}행)')
    print(f'값 기준 중복  : labeled {len(v_dup)}행')

    # labeled 기간 안에 unlabeled 행이 섞여 있는지 (같은 날 다른 샷)
    lab_days = set(lab['TimeStamp'].dt.date)
    same_day = UL['TimeStamp'].dt.date.isin(lab_days)
    print(f'labeled와 같은 날짜의 unlabeled 행: {int(same_day.sum()):,}행')
    print()

### 5-3. `ERR_FACT_QTY` 상태

`ERR_FACT_QTY`는 가이드북상 "생산 시 발생한 불량 개수"입니다. 샷마다 다른 값인지, 하루(또는 생산 계획) 단위로 같은 값이 반복되는지 확인합니다.

In [ ]:
for p in ['cn7', 'rg3']:
    UL = pd.read_csv(os.path.join(OUTPUT_DIR, f'linked_unlabeled_{p}.csv'), parse_dates=['TimeStamp'])
    print(f'==================== {p} ====================')
    if 'ERR_FACT_QTY' not in UL:
        print('ERR_FACT_QTY 컬럼 없음'); print(); continue
    q = UL['ERR_FACT_QTY']
    print(f'결측 {q.isna().mean():.1%} | 0인 행 {(q == 0).mean():.1%} | 최소 {q.min()} 최대 {q.max()} | 고유값 {q.nunique()}개')
    UL['ts_date'] = UL['TimeStamp'].dt.date
    for key in [k for k in ['PART_FACT_PLAN_DATE', 'ts_date'] if k in UL]:
        g = UL.groupby(key)['ERR_FACT_QTY'].agg(['size', 'nunique', 'min', 'max'])
        print(f'\n[{key}별] 행 수 / 고유값 수 / 최소 / 최대')
        print(g.to_string())
        if (g['nunique'] <= 1).all():
            print(f'→ {key} 안에서 값이 하나로 고정: {key} 단위로 기록된 불량 수로 보임')
    print()

### 5-4. 확인 결과 정리 (실행 후 작성)

| 항목 | CN7 | RG3 |
|---|---|---|
| 2차 unlabeled 선택 기준 | | |
| labeled ↔ unlabeled 중복 | | |
| `ERR_FACT_QTY` 기록 단위 | | |